# 17 · Height Stability of Minimal Features (Bootstrap Update)

This notebook extends Notebook 17 by adding **bootstrap uncertainty estimates** across height blocks.

## Core question

If a small feature set captures the zeta–GUE correspondence, does it remain effective across later height ranges, and are those trends stable under resampling?

## Feature sets tracked

We focus on the strongest low-dimensional feature sets found earlier:

1. **unevenness**
2. **entropy + unevenness**
3. **entropy + unevenness + ratio_mean**

## What this notebook measures

For each height block, we compare zeta, Poisson, and GUE using:
- centroid gap
- density-grid gap
- Wasserstein gaps on PC1 and PC2

and now also compute:
- bootstrap mean
- bootstrap standard deviation
- bootstrap 95% interval

Positive gaps mean:
\[
d(\text{zeta}, \text{Poisson}) > d(\text{zeta}, \text{GUE}),
\]
so larger is stronger.

In [ ]:
import numpy as np
import mpmath as mp
import matplotlib.pyplot as plt

mp.mp.dps = 50
rng = np.random.default_rng(9423)

## Base data

In [ ]:
N = 1800
zeros = [mp.zetazero(n) for n in range(1, N + 1)]
t = np.array([float(mp.im(z)) for z in zeros])
zeta_gaps_full = np.diff(t)

poisson_gaps_full = rng.exponential(scale=1.0, size=len(zeta_gaps_full))

def sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    all_gaps = []
    for _ in range(n_matrices):
        real = rng.normal(size=(matrix_size, matrix_size))
        imag = rng.normal(size=(matrix_size, matrix_size))
        A = real + 1j * imag
        H = (A + A.conj().T) / 2.0
        eigvals = np.linalg.eigvalsh(H)
        eigvals = np.sort(eigvals)

        n = len(eigvals)
        keep = int(n * bulk_fraction)
        start = (n - keep) // 2
        stop = start + keep

        bulk_vals = eigvals[start:stop]
        bulk_gaps = np.diff(bulk_vals)
        all_gaps.extend(bulk_gaps.tolist())

    return np.array(all_gaps)

gue_gaps_full = sample_gue_bulk_spacings(matrix_size=150, n_matrices=70, bulk_fraction=0.5, rng=rng)

len(zeta_gaps_full), len(poisson_gaps_full), len(gue_gaps_full)

## Window and feature helpers

In [ ]:
def extract_windows(gaps, k=5):
    return np.array([gaps[i:i+k] for i in range(len(gaps) - k + 1)])

def normalize_window(window):
    w = np.array(window, dtype=float)
    return w / w.mean()

def unevenness(window):
    return float(np.max(window) - np.min(window))

def reversal_symmetry_score(window):
    return float(np.mean(np.abs(window - window[::-1])))

def window_entropy(window):
    eps = 1e-12
    p = window / np.sum(window)
    return float(-np.sum(p * np.log(p + eps)))

def ratio_mean(window):
    g1 = window[:-1]
    g2 = window[1:]
    r = np.minimum(g1, g2) / np.maximum(g1, g2)
    return float(np.mean(r))

feature_map = {
    "unevenness": unevenness,
    "entropy": window_entropy,
    "ratio_mean": ratio_mean,
    "symmetry": reversal_symmetry_score,
}

def build_feature_matrix(gaps, feature_names, k=5):
    windows = extract_windows(gaps, k=k)
    windows_n = np.array([normalize_window(w) for w in windows])
    X = np.array([
        [feature_map[name](w) for name in feature_names]
        for w in windows_n
    ], dtype=float)
    return windows_n, X

## Embedding and distance helpers

In [ ]:
def pca_embed_three(A, B, C):
    all_desc = np.vstack([A, B, C])
    mean = all_desc.mean(axis=0)
    std = all_desc.std(axis=0)
    std[std == 0] = 1.0

    A_std = (A - mean) / std
    B_std = (B - mean) / std
    C_std = (C - mean) / std

    X = np.vstack([A_std, B_std, C_std])
    Xc = X - X.mean(axis=0)

    if Xc.shape[1] == 1:
        emb = np.column_stack([Xc[:, 0], np.zeros(len(Xc))])
    else:
        cov = np.cov(Xc, rowvar=False)
        eigvals, eigvecs = np.linalg.eigh(cov)
        order = np.argsort(eigvals)[::-1]
        eigvecs = eigvecs[:, order]
        comps = eigvecs[:, :2]
        emb = Xc @ comps

    nA, nB, nC = len(A), len(B), len(C)
    return emb[:nA], emb[nA:nA+nB], emb[nA+nB:]

def centroid_distance(A, B):
    return float(np.linalg.norm(A.mean(axis=0) - B.mean(axis=0)))

def density_grid(X, bins_x, bins_y):
    H, _, _ = np.histogram2d(X[:,0], X[:,1], bins=[bins_x, bins_y], density=True)
    return H

def grid_l2_distance(H1, H2):
    return float(np.linalg.norm(H1.ravel() - H2.ravel()))

def wasserstein_1d(a, b):
    a = np.sort(np.asarray(a))
    b = np.sort(np.asarray(b))
    n = min(len(a), len(b))
    if len(a) != n:
        idx = np.linspace(0, len(a) - 1, n).astype(int)
        a = a[idx]
    if len(b) != n:
        idx = np.linspace(0, len(b) - 1, n).astype(int)
        b = b[idx]
    return float(np.mean(np.abs(a - b)))

def compute_feature_set_metrics(zeta_X, poisson_X, gue_X):
    zeta_emb, poisson_emb, gue_emb = pca_embed_three(zeta_X, poisson_X, gue_X)

    xmin = min(zeta_emb[:,0].min(), poisson_emb[:,0].min(), gue_emb[:,0].min())
    xmax = max(zeta_emb[:,0].max(), poisson_emb[:,0].max(), gue_emb[:,0].max())
    ymin = min(zeta_emb[:,1].min(), poisson_emb[:,1].min(), gue_emb[:,1].min())
    ymax = max(zeta_emb[:,1].max(), poisson_emb[:,1].max(), gue_emb[:,1].max())

    if xmin == xmax:
        xmin, xmax = xmin - 1, xmax + 1
    if ymin == ymax:
        ymin, ymax = ymin - 1, ymax + 1

    bins_x = np.linspace(xmin, xmax, 24)
    bins_y = np.linspace(ymin, ymax, 24)

    zeta_H = density_grid(zeta_emb, bins_x, bins_y)
    poisson_H = density_grid(poisson_emb, bins_x, bins_y)
    gue_H = density_grid(gue_emb, bins_x, bins_y)

    cd_zg = centroid_distance(zeta_emb, gue_emb)
    cd_zp = centroid_distance(zeta_emb, poisson_emb)

    gd_zg = grid_l2_distance(zeta_H, gue_H)
    gd_zp = grid_l2_distance(zeta_H, poisson_H)

    w1_zg = wasserstein_1d(zeta_emb[:,0], gue_emb[:,0])
    w1_zp = wasserstein_1d(zeta_emb[:,0], poisson_emb[:,0])

    w2_zg = wasserstein_1d(zeta_emb[:,1], gue_emb[:,1])
    w2_zp = wasserstein_1d(zeta_emb[:,1], poisson_emb[:,1])

    return {
        "centroid_gap": cd_zp - cd_zg,
        "grid_gap": gd_zp - gd_zg,
        "wasserstein_pc1_gap": w1_zp - w1_zg,
        "wasserstein_pc2_gap": w2_zp - w2_zg,
        "zeta_emb": zeta_emb,
        "poisson_emb": poisson_emb,
        "gue_emb": gue_emb,
    }

## Bootstrap helper

In [ ]:
def bootstrap_rows(X, rng):
    idx = rng.integers(0, len(X), size=len(X))
    return X[idx]

def bootstrap_feature_set_metrics(zeta_X, poisson_X, gue_X, n_boot=100, seed=9423):
    boot_rng = np.random.default_rng(seed)

    records = []
    for _ in range(n_boot):
        zb = bootstrap_rows(zeta_X, boot_rng)
        pb = bootstrap_rows(poisson_X, boot_rng)
        gb = bootstrap_rows(gue_X, boot_rng)
        records.append(compute_feature_set_metrics(zb, pb, gb))

    summary = {}
    for metric_name in ["centroid_gap", "grid_gap", "wasserstein_pc1_gap", "wasserstein_pc2_gap"]:
        vals = np.array([r[metric_name] for r in records])
        summary[metric_name] = {
            "mean": float(vals.mean()),
            "std": float(vals.std()),
            "q025": float(np.quantile(vals, 0.025)),
            "q975": float(np.quantile(vals, 0.975)),
        }
    return summary

## Height blocks and tracked feature sets

In [ ]:
block_size = 300
block_starts = [0, 300, 600, 900, 1200]

feature_sets = {
    "unevenness": ("unevenness",),
    "entropy+unevenness": ("entropy", "unevenness"),
    "entropy+unevenness+ratio_mean": ("entropy", "unevenness", "ratio_mean"),
}

block_labels = [f"{s+1}-{s+block_size}" for s in block_starts]
block_labels

## Compute blockwise metrics and bootstrap intervals

In [ ]:
results = []

for block_start in block_starts:
    zeta_block = zeta_gaps_full[block_start:block_start + block_size]
    poisson_block = poisson_gaps_full[block_start:block_start + block_size]
    gue_block = gue_gaps_full[:max(block_size + 20, 500)]

    for name, feat_set in feature_sets.items():
        _, zeta_X = build_feature_matrix(zeta_block, feat_set, k=5)
        _, poisson_X = build_feature_matrix(poisson_block, feat_set, k=5)
        _, gue_X = build_feature_matrix(gue_block, feat_set, k=5)

        m = min(len(zeta_X), len(poisson_X), len(gue_X))
        zeta_X = zeta_X[:m]
        poisson_X = poisson_X[:m]
        gue_X = gue_X[:m]

        point_metrics = compute_feature_set_metrics(zeta_X, poisson_X, gue_X)
        boot_summary = bootstrap_feature_set_metrics(
            zeta_X, poisson_X, gue_X, n_boot=100, seed=9423 + block_start
        )

        results.append({
            "block_start": block_start,
            "block_label": f"{block_start+1}-{block_start+block_size}",
            "feature_set_name": name,
            "feature_set": feat_set,
            "point_metrics": point_metrics,
            "bootstrap_summary": boot_summary,
            "zeta_emb": point_metrics["zeta_emb"],
            "poisson_emb": point_metrics["poisson_emb"],
            "gue_emb": point_metrics["gue_emb"],
        })

len(results)

## Raw bootstrap summaries

In [ ]:
results[:2]

## Centroid-gap stability across height with bootstrap error bars

In [ ]:
plt.figure(figsize=(9.5, 5.2))
x = np.arange(len(block_labels))

for name in feature_sets:
    rows = [r for r in results if r["feature_set_name"] == name]
    means = np.array([r["bootstrap_summary"]["centroid_gap"]["mean"] for r in rows])
    lows = np.array([r["bootstrap_summary"]["centroid_gap"]["mean"] - r["bootstrap_summary"]["centroid_gap"]["q025"] for r in rows])
    highs = np.array([r["bootstrap_summary"]["centroid_gap"]["q975"] - r["bootstrap_summary"]["centroid_gap"]["mean"] for r in rows])

    plt.errorbar(
        x, means, yerr=np.vstack([lows, highs]),
        marker="o", capsize=4, label=name
    )

plt.axhline(0.0, linestyle="--", linewidth=1)
plt.xticks(x, block_labels)
plt.ylabel("centroid gap")
plt.title("Centroid-gap stability across height (bootstrap)")
plt.legend()
plt.show()

## Density-grid-gap stability across height with bootstrap error bars

In [ ]:
plt.figure(figsize=(9.5, 5.2))
x = np.arange(len(block_labels))

for name in feature_sets:
    rows = [r for r in results if r["feature_set_name"] == name]
    means = np.array([r["bootstrap_summary"]["grid_gap"]["mean"] for r in rows])
    lows = np.array([r["bootstrap_summary"]["grid_gap"]["mean"] - r["bootstrap_summary"]["grid_gap"]["q025"] for r in rows])
    highs = np.array([r["bootstrap_summary"]["grid_gap"]["q975"] - r["bootstrap_summary"]["grid_gap"]["mean"] for r in rows])

    plt.errorbar(
        x, means, yerr=np.vstack([lows, highs]),
        marker="o", capsize=4, label=name
    )

plt.axhline(0.0, linestyle="--", linewidth=1)
plt.xticks(x, block_labels)
plt.ylabel("grid L2 gap")
plt.title("Density-grid-gap stability across height (bootstrap)")
plt.legend()
plt.show()

## Wasserstein stability across height with bootstrap error bars

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharex=True)
x = np.arange(len(block_labels))

for name in feature_sets:
    rows = [r for r in results if r["feature_set_name"] == name]

    means1 = np.array([r["bootstrap_summary"]["wasserstein_pc1_gap"]["mean"] for r in rows])
    lows1 = np.array([r["bootstrap_summary"]["wasserstein_pc1_gap"]["mean"] - r["bootstrap_summary"]["wasserstein_pc1_gap"]["q025"] for r in rows])
    highs1 = np.array([r["bootstrap_summary"]["wasserstein_pc1_gap"]["q975"] - r["bootstrap_summary"]["wasserstein_pc1_gap"]["mean"] for r in rows])

    means2 = np.array([r["bootstrap_summary"]["wasserstein_pc2_gap"]["mean"] for r in rows])
    lows2 = np.array([r["bootstrap_summary"]["wasserstein_pc2_gap"]["mean"] - r["bootstrap_summary"]["wasserstein_pc2_gap"]["q025"] for r in rows])
    highs2 = np.array([r["bootstrap_summary"]["wasserstein_pc2_gap"]["q975"] - r["bootstrap_summary"]["wasserstein_pc2_gap"]["mean"] for r in rows])

    axes[0].errorbar(x, means1, yerr=np.vstack([lows1, highs1]), marker="o", capsize=4, label=name)
    axes[1].errorbar(x, means2, yerr=np.vstack([lows2, highs2]), marker="o", capsize=4, label=name)

axes[0].axhline(0.0, linestyle="--", linewidth=1)
axes[1].axhline(0.0, linestyle="--", linewidth=1)

axes[0].set_xticks(x, block_labels)
axes[1].set_xticks(x, block_labels)

axes[0].set_ylabel("Wasserstein PC1 gap")
axes[1].set_ylabel("Wasserstein PC2 gap")
axes[0].set_title("PC1 stability across height (bootstrap)")
axes[1].set_title("PC2 stability across height (bootstrap)")

axes[0].legend()
plt.tight_layout()
plt.show()

## Blockwise bootstrap summary table

In [ ]:
blockwise_summary = {}
for name in feature_sets:
    rows = [r for r in results if r["feature_set_name"] == name]
    blockwise_summary[name] = {
        "centroid_gap_mean": float(np.mean([r["bootstrap_summary"]["centroid_gap"]["mean"] for r in rows])),
        "centroid_gap_min_q025": float(np.min([r["bootstrap_summary"]["centroid_gap"]["q025"] for r in rows])),
        "grid_gap_mean": float(np.mean([r["bootstrap_summary"]["grid_gap"]["mean"] for r in rows])),
        "grid_gap_min_q025": float(np.min([r["bootstrap_summary"]["grid_gap"]["q025"] for r in rows])),
        "wasserstein_pc1_gap_mean": float(np.mean([r["bootstrap_summary"]["wasserstein_pc1_gap"]["mean"] for r in rows])),
        "wasserstein_pc1_gap_min_q025": float(np.min([r["bootstrap_summary"]["wasserstein_pc1_gap"]["q025"] for r in rows])),
        "wasserstein_pc2_gap_mean": float(np.mean([r["bootstrap_summary"]["wasserstein_pc2_gap"]["mean"] for r in rows])),
        "wasserstein_pc2_gap_min_q025": float(np.min([r["bootstrap_summary"]["wasserstein_pc2_gap"]["q025"] for r in rows])),
    }

blockwise_summary

## Representative embeddings by height

Show the strongest 3-feature set across several zeta blocks.

In [ ]:
target_name = "entropy+unevenness+ratio_mean"
target_rows = [r for r in results if r["feature_set_name"] == target_name]

fig, axes = plt.subplots(1, len(target_rows), figsize=(18, 4.2))

for ax, row in zip(axes, target_rows):
    ax.scatter(row["zeta_emb"][:,0], row["zeta_emb"][:,1], s=8, alpha=0.22, label="zeta")
    ax.scatter(row["poisson_emb"][:,0], row["poisson_emb"][:,1], s=8, alpha=0.22, label="Poisson")
    ax.scatter(row["gue_emb"][:,0], row["gue_emb"][:,1], s=8, alpha=0.22, label="GUE")
    ax.set_title(row["block_label"])
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

axes[0].legend()
plt.tight_layout()
plt.show()

## Final summary

In [ ]:
summary = {
    "block_labels": block_labels,
    "feature_sets": feature_sets,
    "blockwise_summary": blockwise_summary,
    "all_results": results,
}
summary

## Notes

- If the main minimal feature sets keep positive bootstrap intervals across multiple height blocks, that supports the idea that the local zeta–GUE correspondence is not just a low-height artifact.
- The 1-feature, 2-feature, and 3-feature sets can behave differently with height; bootstrap uncertainty helps show which are most stable.
- This notebook uses contiguous blocks rather than logarithmic height bands; later versions could compare different sampling schemes.

## Next directions

A natural next notebook could do one of these:

1. compare contiguous blocks with random subsamples of equal size  
2. extend the same analysis to conditional windows (after small vs after large gaps)  
3. compare how classifier accuracy varies with height for the same minimal feature sets  
4. test whether the bootstrap intervals stay positive under different block sizes